In [3]:
import pandas as pd
df = pd.read_csv("missed_class.csv")
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [4]:
df["rating"].unique()

array([3, 5, 4, 2, 1])

In [5]:
df = df[["rating","reviewText"]]
df.head()

,rating,reviewText
0,3,"Jace Rankin may be short, but he's nothing to ..."
1,5,Great short read. I didn't want to put it dow...
2,3,I'll start by saying this is the first of four...
3,3,Aggie is Angela Lansbury who carries pocketboo...
4,4,I did not expect this type of book to be in li...


In [6]:
df["rating"] = df["rating"].apply(lambda x: 0 if x < 4 else 1)
df.head()

,rating,reviewText
0,0,"Jace Rankin may be short, but he's nothing to ..."
1,1,Great short read. I didn't want to put it dow...
2,0,I'll start by saying this is the first of four...
3,0,Aggie is Angela Lansbury who carries pocketboo...
4,1,I did not expect this type of book to be in li...


In [7]:
df["rating"].value_counts()

rating
0    6000
1    6000
Name: count, dtype: int64

In [8]:
df = df.rename(columns={"reviewText":"review"})
X = df["review"]
y = df["rating"]

In [9]:
df.head()

,rating,review
0,0,"Jace Rankin may be short, but he's nothing to ..."
1,1,Great short read. I didn't want to put it dow...
2,0,I'll start by saying this is the first of four...
3,0,Aggie is Angela Lansbury who carries pocketboo...
4,1,I did not expect this type of book to be in li...


In [10]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer


In [11]:
stopwords = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [12]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9]", " ", text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word)for word in tokens if word not in stopwords]
    return " ".join(tokens)

In [13]:
df.head()

,rating,review
0,0,"Jace Rankin may be short, but he's nothing to ..."
1,1,Great short read. I didn't want to put it dow...
2,0,I'll start by saying this is the first of four...
3,0,Aggie is Angela Lansbury who carries pocketboo...
4,1,I did not expect this type of book to be in li...


In [15]:
X_clean = X.apply(preprocess_text)
X_clean

0        jace rankin may short nothing mess man hauled ...
1        great short read want put read one sitting sex...
2        start saying first four book expecting 34 conc...
3        aggie angela lansbury carry pocketbook instead...
4        expect type book library pleased find price right
                               ...                        
11995    valentine cupid vampire jena ian another vampi...
11996    read seven book series apocalyptic adventure o...
11997    book really cuppa situation man capturing woma...
11998    tried use charge kindle even register charging...
11999    taking instruction look often hidden world sex...
Name: review, Length: 12000, dtype: object

In [16]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features= 5000,ngram_range=(1, 2) )

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

In [21]:
# from sklearn.linear_model import LogisticRegression
#
# model = LogisticRegression(max_iter=1000)
#
#
# model.fit(X_train_tfidf, y_train)

In [22]:
from sklearn.naive_bayes import MultinomialNB
model1 = MultinomialNB()
model1.fit(X_train_tfidf, y_train)


,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [24]:
y_pred= model1.predict(X_test_tfidf)

In [26]:
from sklearn.metrics import accuracy_score , confusion_matrix , classification_report
print("Accuracy Score : {}".format(accuracy_score(y_test, y_pred)))
print("confusion matrix : {}".format(confusion_matrix(y_test, y_pred)))
print("Classification Report : {}".format(classification_report(y_test, y_pred)))

Accuracy Score : 0.83625
confusion matrix : [[1009  191]
 [ 202  998]]
Classification Report :               precision    recall  f1-score   support

           0       0.83      0.84      0.84      1200
           1       0.84      0.83      0.84      1200

    accuracy                           0.84      2400
   macro avg       0.84      0.84      0.84      2400
weighted avg       0.84      0.84      0.84      2400



In [27]:
df.head(8)

,rating,review
0,0,"Jace Rankin may be short, but he's nothing to ..."
1,1,Great short read. I didn't want to put it dow...
2,0,I'll start by saying this is the first of four...
3,0,Aggie is Angela Lansbury who carries pocketboo...
4,1,I did not expect this type of book to be in li...
5,1,Aislinn is a little girl with big dreams. Afte...
6,0,This has the makings of a good story... unfort...
7,1,I got this because I like collaborated short s...


In [29]:
message = "I dislike the book , everything about it seems off.i wont recomment it for anyon"


In [32]:
message = preprocess_text(message)

In [35]:
message = tfidf.transform([message])

AttributeError: 'csr_matrix' object has no attribute 'lower'

In [37]:
pred = model1.predict(message)

In [148]:
print(pred[0])

0


In [38]:
import pickle

with open("mode1.pkl", "wb") as f:
    pickle.dump(model1, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
